<a href="https://colab.research.google.com/github/lizlorenzana931109-art/FinPredict-Actividad5/blob/main/fuentes/03_modelado_experimentacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# FinPredict | Actividad 5
## 03. Modelado y experimentación

### Objetivo

Desarrollar, optimizar y comparar modelos supervisados de Machine Learning para predecir el incumplimiento de pago hipotecario a 90 días.

### Modelos seleccionados

1. **Regresión logística:** modelo de referencia interpretable.
2. **Random Forest:** modelo de ensamble capaz de identificar relaciones no lineales.

### Metodología

- Carga del dataset preparado `v1`.
- Separación temporal entre desarrollo y evaluación final.
- Construcción de matrices predictoras.
- Entrenamiento de modelos base.
- Optimización mediante GridSearchCV.
- Registro de experimentos con MLflow.
- Comparación de métricas y selección del modelo.
- Evaluación final sobre el periodo reservado.

### Métricas

Se evaluarán:

- Precision.
- Recall.
- F1-score.
- ROC-AUC.
- PR-AUC.
- Matriz de confusión.

En riesgo crediticio, prestaremos especial atención al **recall de incumplimiento**, sin ignorar el costo operativo de los falsos positivos.

### Controles metodológicos

- Utilizar únicamente las 14 variables predictoras seleccionadas.
- Excluir identificadores y columnas de auditoría del entrenamiento.
- Mantener reservada la evaluación final.
- Evitar fuga de información durante el preprocesamiento y la validación temporal.
- Documentar parámetros, resultados y decisiones de selección.

**Proyecto:** FinPredict  
**Etapa:** Fase 2 — Modelado y experimentación  
**Herramientas:** Python, scikit-learn, MLflow y Google Colab


In [1]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 2
# Carga y verificación del conjunto de desarrollo
# ============================================================

import pandas as pd
import numpy as np
import requests
import hashlib
from io import BytesIO

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "lizlorenzana931109-art/"
    "FinPredict-Actividad5/main/datos/datos_limp/"
)

# Descargar manifiesto de la versión v1
respuesta = requests.get(
    BASE_URL + "manifiesto_v1.json",
    timeout=60
)
respuesta.raise_for_status()

manifiesto = respuesta.json()

# Descargar únicamente el conjunto de desarrollo
nombre_archivo = "finpredict_desarrollo_v1.parquet"

respuesta = requests.get(
    BASE_URL + nombre_archivo,
    timeout=180
)
respuesta.raise_for_status()

contenido = respuesta.content

# Validar integridad contra el manifiesto
sha256 = hashlib.sha256(contenido).hexdigest()

assert sha256 == manifiesto["archivos"][nombre_archivo]["sha256"], (
    "La huella SHA-256 no coincide."
)

# Cargar datos
df_desarrollo = pd.read_parquet(BytesIO(contenido))

# Recuperar selección de predictores
PREDICTORES = manifiesto["predictores"]
VARIABLE_OBJETIVO = manifiesto["variable_objetivo"]

# Controles básicos
assert len(df_desarrollo) == 350_520
assert len(PREDICTORES) == 14
assert df_desarrollo["uso"].eq("HISTORICO").all()
assert not df_desarrollo[PREDICTORES].isna().any().any()

print("FINPREDICT | Carga del conjunto de desarrollo")
print("-" * 60)

print(f"Observaciones: {len(df_desarrollo):,}")
print(f"Columnas: {df_desarrollo.shape[1]}")
print(f"Predictores: {len(PREDICTORES)}")
print(f"Eventos de incumplimiento: {df_desarrollo[VARIABLE_OBJETIVO].sum():,}")
print(f"Integridad SHA-256: VERIFICADA")

print("-" * 60)
print("Datos de desarrollo cargados correctamente.")


FINPREDICT | Carga del conjunto de desarrollo
------------------------------------------------------------
Observaciones: 350,520
Columnas: 34
Predictores: 14
Eventos de incumplimiento: 21,368
Integridad SHA-256: VERIFICADA
------------------------------------------------------------
Datos de desarrollo cargados correctamente.


In [2]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 3
# Definición de validación temporal expansiva
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import PredefinedSplit

# Ordenar el histórico sin modificar la fuente original
df_modelado = (
    df_desarrollo
    .sort_values(["fecha_corte", "id_credito"])
    .reset_index(drop=True)
)

# Cortes históricos disponibles
cortes = sorted(df_modelado["fecha_corte"].unique())

# Elegimos tres ventanas de validación
cortes_validacion = cortes[-3:]

particiones = []

for corte_validacion in cortes_validacion:

    # Solo entrenar con observaciones cuyas
    # etiquetas ya eran conocidas al corte de validación
    mascara_train = (
        df_modelado["fin_prediccion"] < corte_validacion
    )

    mascara_validacion = (
        df_modelado["fecha_corte"] == corte_validacion
    )

    idx_train = np.flatnonzero(mascara_train)
    idx_validacion = np.flatnonzero(mascara_validacion)

    assert len(idx_train) > 0
    assert len(idx_validacion) > 0

    assert (
        df_modelado.loc[
            idx_train, "fin_prediccion"
        ].max()
        < corte_validacion
    )

    particiones.append((idx_train, idx_validacion))

# Resumen de particiones
resumen_particiones = pd.DataFrame([
    {
        "fold": i,
        "ultimo_corte_train": df_modelado.loc[
            idx_train, "fecha_corte"
        ].max().date(),
        "corte_validacion": df_modelado.loc[
            idx_validacion, "fecha_corte"
        ].iloc[0].date(),
        "registros_train": len(idx_train),
        "registros_validacion": len(idx_validacion),
        "eventos_validacion": int(
            df_modelado.loc[
                idx_validacion, VARIABLE_OBJETIVO
            ].sum()
        )
    }
    for i, (idx_train, idx_validacion)
    in enumerate(particiones, start=1)
])

print("FINPREDICT | Validación temporal")
print("-" * 65)

display(resumen_particiones)

print("-" * 65)
print("Particiones temporales construidas.")
print("Evaluación final 2025-Q3 permanece reservada.")


FINPREDICT | Validación temporal
-----------------------------------------------------------------


,fold,ultimo_corte_train,corte_validacion,registros_train,registros_validacion,eventos_validacion
0,1,2024-06-30,2024-12-31,210863,33792,1848
1,2,2024-09-30,2025-03-31,242453,36111,2116
2,3,2024-12-31,2025-06-30,276245,38164,2252


-----------------------------------------------------------------
Particiones temporales construidas.
Evaluación final 2025-Q3 permanece reservada.


In [3]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 4
# Preparación de entradas para modelado
# ============================================================

import numpy as np
import pandas as pd

# Conservar intacto el dataset cargado
df_entrenamiento = df_modelado.copy(deep=True)

# Identificar ingresos imputados estadísticamente en v1
mascara_mediana = (
    df_entrenamiento["metodo_recuperacion_ingreso"]
    == "MEDIANA_TIPO_EMPLEO"
)

registros_mediana = int(mascara_mediana.sum())

# Retirar la imputación estadística global
df_entrenamiento.loc[
    mascara_mediana, "ingreso_mensual"
] = np.nan

# Retirar también la relación derivada de esos ingresos
df_entrenamiento.loc[
    mascara_mediana, "relacion_pago_ingreso"
] = np.nan

# Construir matrices sin incluir auditoría ni identificadores
X = df_entrenamiento[PREDICTORES].copy()

y = df_entrenamiento[VARIABLE_OBJETIVO].copy()

# Validaciones
assert len(X) == len(y) == 350_520

assert X["ingreso_mensual"].isna().sum() == registros_mediana
assert X["relacion_pago_ingreso"].isna().sum() == registros_mediana

assert y.isin([0, 1]).all()

print("FINPREDICT | Preparación para modelado")
print("-" * 60)

print(f"Observaciones: {len(X):,}")
print(f"Predictores: {X.shape[1]}")
print(f"Ingresos pendientes de imputación por fold: {registros_mediana}")
print(f"Relaciones pago-ingreso pendientes: {X['relacion_pago_ingreso'].isna().sum():,}")

print("-" * 60)
print("Matrices preparadas. No se ha entrenado ningún modelo.")


FINPREDICT | Preparación para modelado
------------------------------------------------------------
Observaciones: 350,520
Predictores: 14
Ingresos pendientes de imputación por fold: 41
Relaciones pago-ingreso pendientes: 41
------------------------------------------------------------
Matrices preparadas. No se ha entrenado ningún modelo.



### 5. Pipeline de preprocesamiento

El preprocesamiento se integrará a los modelos mediante `Pipeline` de scikit-learn.

Esto permitirá ajustar los parámetros de imputación exclusivamente con los registros disponibles en cada conjunto de entrenamiento durante la validación temporal.

La transformación contempla la dependencia matemática entre `ingreso_mensual` y `relacion_pago_ingreso`, recalculando esta última cuando el ingreso sea imputado.

Para la regresión logística se aplicará posteriormente estandarización de variables mediante `StandardScaler`. Para Random Forest no será necesaria.


In [4]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 5
# Transformador de imputación y consistencia financiera
# ============================================================

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


class ImputadorFinPredict(BaseEstimator, TransformerMixin):
    """
    Imputa ingreso_mensual con la mediana calculada
    durante fit y recalcula relacion_pago_ingreso
    únicamente para los registros imputados.
    """

    def fit(self, X, y=None):

        X = X.copy()

        self.mediana_ingreso_ = X[
            "ingreso_mensual"
        ].median()

        if pd.isna(self.mediana_ingreso_):
            raise ValueError(
                "No hay ingresos válidos para calcular la mediana."
            )

        if self.mediana_ingreso_ <= 0:
            raise ValueError(
                "La mediana del ingreso debe ser positiva."
            )

        return self

    def transform(self, X):

        X = X.copy()

        # Identificar únicamente los ingresos faltantes
        mascara_nulos = X["ingreso_mensual"].isna()

        # Aplicar mediana aprendida en fit
        X["ingreso_mensual"] = (
            X["ingreso_mensual"]
            .fillna(self.mediana_ingreso_)
        )

        # Recalcular relación financiera de esos registros
        X.loc[
            mascara_nulos, "relacion_pago_ingreso"
        ] = (
            X.loc[mascara_nulos, "pago_mensual"]
            / X.loc[mascara_nulos, "ingreso_mensual"]
        )

        # Verificar ausencia de nulos e infinitos
        if X.isna().any().any():
            raise ValueError(
                "Persisten valores faltantes después del tratamiento."
            )

        if not np.isfinite(
            X.to_numpy(dtype=float)
        ).all():
            raise ValueError(
                "Se encontraron valores infinitos."
            )

        return X


# Probar el transformador sin entrenar modelos
transformador_prueba = ImputadorFinPredict()

# Solo para verificar funcionamiento:
# el ajuste definitivo se hará dentro de cada fold
X_prueba = transformador_prueba.fit_transform(X)

assert X_prueba.isna().sum().sum() == 0

assert np.isfinite(
    X_prueba.to_numpy(dtype=float)
).all()

assert len(X_prueba) == len(X)

print("FINPREDICT | Preprocesamiento")
print("-" * 60)

print(
    f"Mediana de prueba: "
    f"${transformador_prueba.mediana_ingreso_:,.2f}"
)

print(
    f"Valores faltantes restantes: "
    f"{X_prueba.isna().sum().sum()}"
)

print(f"Variables conservadas: {X_prueba.shape[1]}")

print("-" * 60)
print("Transformador validado.")
print("Los modelos todavía no han sido entrenados.")


FINPREDICT | Preprocesamiento
------------------------------------------------------------
Mediana de prueba: $85,978.11
Valores faltantes restantes: 0
Variables conservadas: 14
------------------------------------------------------------
Transformador validado.
Los modelos todavía no han sido entrenados.



### 6. Modelo base: regresión logística

Se entrena un modelo de regresión logística como referencia inicial para la predicción del incumplimiento hipotecario.

El modelo utiliza un Pipeline que integra:

1. Imputación de ingresos y conciliación de la relación pago-ingreso.
2. Estandarización de variables numéricas.
3. Regresión logística con ponderación de clases.

El primer experimento utiliza una partición temporal y se evalúa mediante Precision, Recall, F1-score, ROC-AUC y PR-AUC.

Esta evaluación es preliminar. La selección definitiva del modelo requerirá validación temporal y optimización de hiperparámetros.


In [5]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 6
# Regresión logística | Modelo base
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

import pandas as pd
import numpy as np

# 1. Primera partición temporal
idx_train, idx_validacion = particiones[0]

X_train = X.iloc[idx_train].copy()
y_train = y.iloc[idx_train].copy()

X_val = X.iloc[idx_validacion].copy()
y_val = y.iloc[idx_validacion].copy()

# 2. Pipeline completo
pipeline_logistico = Pipeline([
    ("imputacion", ImputadorFinPredict()),
    ("escalado", StandardScaler()),
    ("modelo", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="lbfgs",
        random_state=2028
    ))
])

# 3. Entrenamiento
pipeline_logistico.fit(X_train, y_train)

# 4. Predicciones sobre validación
probabilidades = pipeline_logistico.predict_proba(X_val)[:, 1]

# Umbral inicial de clasificación
UMBRAL_BASE = 0.50

predicciones = (
    probabilidades >= UMBRAL_BASE
).astype(int)

# 5. Evaluación
metricas_logistica_base = {
    "Precision": precision_score(
        y_val, predicciones, zero_division=0
    ),
    "Recall": recall_score(
        y_val, predicciones, zero_division=0
    ),
    "F1-score": f1_score(
        y_val, predicciones, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_val, probabilidades
    ),
    "PR-AUC": average_precision_score(
        y_val, probabilidades
    )
}

resumen_metricas = pd.DataFrame(
    metricas_logistica_base.items(),
    columns=["metrica", "valor"]
)

matriz_confusion = confusion_matrix(
    y_val,
    predicciones,
    labels=[0, 1]
)

print("FINPREDICT | Regresión logística - Modelo base")
print("-" * 65)

print(f"Entrenamiento: {len(X_train):,}")
print(f"Validación: {len(X_val):,}")
print(f"Incumplimientos reales: {int(y_val.sum()):,}")
print(f"Umbral: {UMBRAL_BASE:.2f}")

print("\nMétricas de validación:")
display(resumen_metricas.round(4))

print("\nMatriz de confusión:")
display(pd.DataFrame(
    matriz_confusion,
    index=["Real: sin incumplimiento", "Real: incumplimiento"],
    columns=[
        "Predicho: sin incumplimiento",
        "Predicho: incumplimiento"
    ]
))

print("-" * 65)
print("Primer entrenamiento completado.")
print("Evaluación final reservada sin utilizar.")


FINPREDICT | Regresión logística - Modelo base
-----------------------------------------------------------------
Entrenamiento: 210,863
Validación: 33,792
Incumplimientos reales: 1,848
Umbral: 0.50

Métricas de validación:


,metrica,valor
0,Precision,0.5217
1,Recall,0.9351
2,F1-score,0.6698
3,ROC-AUC,0.9859
4,PR-AUC,0.8547



Matriz de confusión:


,Predicho: sin incumplimiento,Predicho: incumplimiento
Real: sin incumplimiento,30360,1584
Real: incumplimiento,120,1728


-----------------------------------------------------------------
Primer entrenamiento completado.
Evaluación final reservada sin utilizar.


In [6]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 7
# Random Forest | Modelo base
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)
import pandas as pd
import time

# 1. Pipeline del modelo
# Random Forest no requiere estandarización.
pipeline_rf = Pipeline([
    ("imputacion", ImputadorFinPredict()),
    ("modelo", RandomForestClassifier(
        n_estimators=120,
        max_depth=12,
        min_samples_leaf=20,
        class_weight="balanced",
        n_jobs=-1,
        random_state=2028
    ))
])

# 2. Entrenamiento: mismo fold que regresión logística
inicio = time.perf_counter()

pipeline_rf.fit(X_train, y_train)

tiempo_rf = time.perf_counter() - inicio

# 3. Predicciones
probabilidades_rf = pipeline_rf.predict_proba(X_val)[:, 1]

predicciones_rf = (
    probabilidades_rf >= UMBRAL_BASE
).astype(int)

# 4. Métricas
metricas_rf_base = {
    "Precision": precision_score(
        y_val, predicciones_rf, zero_division=0
    ),
    "Recall": recall_score(
        y_val, predicciones_rf, zero_division=0
    ),
    "F1-score": f1_score(
        y_val, predicciones_rf, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_val, probabilidades_rf
    ),
    "PR-AUC": average_precision_score(
        y_val, probabilidades_rf
    )
}

# 5. Comparación inicial de modelos
comparativo_base = pd.DataFrame({
    "Regresión logística": metricas_logistica_base,
    "Random Forest": metricas_rf_base
}).round(4)

matriz_rf = confusion_matrix(
    y_val,
    predicciones_rf,
    labels=[0, 1]
)

print("FINPREDICT | Random Forest - Modelo base")
print("-" * 65)

print(f"Entrenamiento: {len(X_train):,}")
print(f"Validación: {len(X_val):,}")
print(f"Tiempo de entrenamiento: {tiempo_rf:.2f} segundos")

print("\nComparación inicial:")
display(comparativo_base)

print("\nMatriz de confusión - Random Forest:")
display(pd.DataFrame(
    matriz_rf,
    index=[
        "Real: sin incumplimiento",
        "Real: incumplimiento"
    ],
    columns=[
        "Predicho: sin incumplimiento",
        "Predicho: incumplimiento"
    ]
))

print("-" * 65)
print("Segundo modelo base entrenado.")
print("Evaluación final permanece reservada.")


FINPREDICT | Random Forest - Modelo base
-----------------------------------------------------------------
Entrenamiento: 210,863
Validación: 33,792
Tiempo de entrenamiento: 66.55 segundos

Comparación inicial:


,Regresión logística,Random Forest
Precision,0.5217,0.5300
Recall,0.9351,0.9324
F1-score,0.6698,0.6758
ROC-AUC,0.9859,0.9860
PR-AUC,0.8547,0.8650



Matriz de confusión - Random Forest:


,Predicho: sin incumplimiento,Predicho: incumplimiento
Real: sin incumplimiento,30416,1528
Real: incumplimiento,125,1723


-----------------------------------------------------------------
Segundo modelo base entrenado.
Evaluación final permanece reservada.


In [7]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 8
# Verificación del esquema de validación para GridSearchCV
# ============================================================

import numpy as np
import pandas as pd

# Verificar los tres folds temporales
registros_validacion = []

for numero, (idx_train, idx_val) in enumerate(
    particiones, start=1
):

    train = df_modelado.iloc[idx_train]
    val = df_modelado.iloc[idx_val]

    # Cada validación debe utilizar un corte distinto
    assert val["fecha_corte"].nunique() == 1

    # Las etiquetas de entrenamiento deben estar
    # disponibles antes del corte de validación
    assert (
        train["fin_prediccion"].max()
        < val["fecha_corte"].min()
    )

    # No debe existir una misma observación
    # contrato-periodo en entrenamiento y validación
    claves_train = set(zip(
        train["id_credito"], train["fecha_corte"]
    ))

    claves_val = set(zip(
        val["id_credito"], val["fecha_corte"]
    ))

    assert claves_train.isdisjoint(claves_val)

    registros_validacion.append({
        "fold": numero,
        "entrenamiento": len(train),
        "validacion": len(val),
        "corte_validacion": val["fecha_corte"].min().date(),
        "eventos": int(val[VARIABLE_OBJETIVO].sum())
    })

resumen_cv = pd.DataFrame(registros_validacion)

print("FINPREDICT | Preparación de GridSearchCV")
print("-" * 65)

display(resumen_cv)

print("-" * 65)
print("Tres folds temporales validados.")
print("Modelos base conservados sin modificaciones.")


FINPREDICT | Preparación de GridSearchCV
-----------------------------------------------------------------


,fold,entrenamiento,validacion,corte_validacion,eventos
0,1,210863,33792,2024-12-31,1848
1,2,242453,36111,2025-03-31,2116
2,3,276245,38164,2025-06-30,2252


-----------------------------------------------------------------
Tres folds temporales validados.
Modelos base conservados sin modificaciones.



### 9. Optimización de regresión logística

Se utiliza `GridSearchCV` para seleccionar hiperparámetros de regresión logística mediante tres particiones temporales expansivas.

**Objetivo de optimización:** maximizar Average Precision (PR-AUC), una métrica adecuada para evaluar la capacidad de ordenamiento de observaciones positivas en un contexto de clases desbalanceadas.

Se comparan distintos niveles de regularización y estrategias de ponderación de clases.

El preprocesamiento forma parte del Pipeline, por lo que la imputación y el escalado se ajustan exclusivamente con los datos de entrenamiento de cada fold.

La evaluación final permanece reservada y no interviene en la selección de hiperparámetros.


In [8]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 9
# GridSearchCV | Regresión logística
# ============================================================

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    make_scorer,
    average_precision_score,
    recall_score,
    f1_score
)

import pandas as pd
import time

# Búsqueda acotada y reproducible
parametros_logistica = {
    "modelo__C": [0.1, 1.0, 10.0],
    "modelo__class_weight": [None, "balanced"]
}

# Métricas de evaluación
metricas_cv = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "recall": "recall",
    "f1": "f1"
}

busqueda_logistica = GridSearchCV(
    estimator=pipeline_logistico,
    param_grid=parametros_logistica,
    scoring=metricas_cv,
    refit="pr_auc",
    cv=particiones,
    n_jobs=2,
    verbose=1,
    return_train_score=False,
    error_score="raise"
)

inicio = time.perf_counter()

busqueda_logistica.fit(X, y)

tiempo_busqueda_logistica = time.perf_counter() - inicio

# Resultados de experimentación
resultados_logistica_cv = pd.DataFrame(
    busqueda_logistica.cv_results_
)

columnas_resultados = [
    "param_modelo__C",
    "param_modelo__class_weight",
    "mean_test_pr_auc",
    "mean_test_roc_auc",
    "mean_test_recall",
    "mean_test_f1",
    "rank_test_pr_auc"
]

resumen_grid_logistica = (
    resultados_logistica_cv[columnas_resultados]
    .sort_values("rank_test_pr_auc")
    .reset_index(drop=True)
)

print("FINPREDICT | GridSearchCV - Regresión logística")
print("-" * 65)

display(resumen_grid_logistica.round(4))

print("\nMejores hiperparámetros:")
print(busqueda_logistica.best_params_)

print(f"\nMejor PR-AUC promedio: {busqueda_logistica.best_score_:.4f}")

print(
    f"Tiempo total de búsqueda: "
    f"{tiempo_busqueda_logistica:.2f} segundos"
)

print("-" * 65)
print("Optimización de regresión logística completada.")


Fitting 3 folds for each of 6 candidates, totalling 18 fits
FINPREDICT | GridSearchCV - Regresión logística
-----------------------------------------------------------------


,param_modelo__C,param_modelo__class_weight,mean_test_pr_auc,mean_test_roc_auc,mean_test_recall,mean_test_f1,rank_test_pr_auc
0,0.1,None,0.8567,0.9844,0.6661,0.7452,1
1,1.0,None,0.8566,0.9844,0.6659,0.7446,2
2,10.0,None,0.8566,0.9844,0.6659,0.7446,3
3,0.1,balanced,0.8492,0.9846,0.9290,0.6807,4
4,1.0,balanced,0.8491,0.9846,0.9290,0.6807,5
5,10.0,balanced,0.8491,0.9846,0.9290,0.6807,6



Mejores hiperparámetros:
{'modelo__C': 0.1, 'modelo__class_weight': None}

Mejor PR-AUC promedio: 0.8567
Tiempo total de búsqueda: 18.83 segundos
-----------------------------------------------------------------
Optimización de regresión logística completada.



### 10. Optimización de Random Forest

Se optimiza el modelo Random Forest mediante GridSearchCV con validación temporal expansiva.

La búsqueda evalúa combinaciones de profundidad máxima y número mínimo de observaciones por hoja, manteniendo constante el número de árboles para controlar el costo computacional.

La métrica principal de selección es Average Precision (PR-AUC). Se registran adicionalmente ROC-AUC, recall y F1-score.

Los resultados se compararán posteriormente con los de regresión logística y se documentarán mediante MLflow.


In [9]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 10
# GridSearchCV | Random Forest
# ============================================================

from sklearn.model_selection import GridSearchCV
import pandas as pd
import time

# Búsqueda acotada para CPU
parametros_rf = {
    "modelo__max_depth": [10, 14],
    "modelo__min_samples_leaf": [10, 30]
}

busqueda_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=parametros_rf,
    scoring=metricas_cv,
    refit="pr_auc",
    cv=particiones,
    n_jobs=2,
    verbose=1,
    return_train_score=False,
    error_score="raise"
)

inicio = time.perf_counter()

busqueda_rf.fit(X, y)

tiempo_busqueda_rf = time.perf_counter() - inicio

# Consolidar resultados
resultados_rf_cv = pd.DataFrame(
    busqueda_rf.cv_results_
)

columnas_rf = [
    "param_modelo__max_depth",
    "param_modelo__min_samples_leaf",
    "mean_test_pr_auc",
    "mean_test_roc_auc",
    "mean_test_recall",
    "mean_test_f1",
    "rank_test_pr_auc"
]

resumen_grid_rf = (
    resultados_rf_cv[columnas_rf]
    .sort_values("rank_test_pr_auc")
    .reset_index(drop=True)
)

print("FINPREDICT | GridSearchCV - Random Forest")
print("-" * 65)

display(resumen_grid_rf.round(4))

print("\nMejores hiperparámetros:")
print(busqueda_rf.best_params_)

print(f"\nMejor PR-AUC promedio: {busqueda_rf.best_score_:.4f}")

print(
    f"Tiempo total de búsqueda: "
    f"{tiempo_busqueda_rf:.2f} segundos"
)

print("-" * 65)
print("Optimización de Random Forest completada.")


Fitting 3 folds for each of 4 candidates, totalling 12 fits
FINPREDICT | GridSearchCV - Random Forest
-----------------------------------------------------------------


,param_modelo__max_depth,param_modelo__min_samples_leaf,mean_test_pr_auc,mean_test_roc_auc,mean_test_recall,mean_test_f1,rank_test_pr_auc
0,14,30,0.8631,0.9855,0.9284,0.6843,1
1,10,30,0.8623,0.9856,0.9307,0.6722,2
2,14,10,0.8618,0.9855,0.9145,0.7020,3
3,10,10,0.8609,0.9855,0.9278,0.6766,4



Mejores hiperparámetros:
{'modelo__max_depth': 14, 'modelo__min_samples_leaf': 30}

Mejor PR-AUC promedio: 0.8631
Tiempo total de búsqueda: 1069.77 segundos
-----------------------------------------------------------------
Optimización de Random Forest completada.



### 11. Seguimiento de experimentos con MLflow

Se incorpora MLflow Tracking para documentar y comparar los experimentos de regresión logística y Random Forest.

Cada ejecución registrará:

- Algoritmo y configuración de hiperparámetros.
- Identificación de la versión del dataset.
- Estrategia de validación temporal.
- Métricas promedio de validación cruzada.
- Tiempo de optimización.
- Artefactos de resultados.

Se conservarán los registros de experimentación para facilitar la trazabilidad, reproducibilidad y auditoría de las decisiones de modelado.

La evaluación final permanece reservada.


In [11]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 11 (CORREGIDA)
# Configuración segura de MLflow
# ============================================================

from pathlib import Path
import os
import sys
import subprocess

# 1. Comprobar que conservamos los entrenamientos
assert "busqueda_logistica" in globals(), (
    "No se encontró la búsqueda de regresión logística."
)

assert "busqueda_rf" in globals(), (
    "No se encontró la búsqueda de Random Forest."
)

print("Modelos optimizados disponibles en memoria.")

# 2. Montar Google Drive si hace falta
from google.colab import drive

if not os.path.ismount("/content/drive_real"):
    drive.mount("/content/drive_real")

assert os.path.ismount("/content/drive_real"), (
    "No se pudo montar Google Drive."
)

# 3. Importar MLflow
try:
    import mlflow
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip",
        "install", "-q", "mlflow"
    ])
    import mlflow

# 4. Almacenamiento activo en disco local de Colab
RUTA_MLFLOW_LOCAL = Path(
    "/content/finpredict_mlflow"
)

RUTA_MLFLOW_LOCAL.mkdir(
    parents=True,
    exist_ok=True
)

RUTA_DB = RUTA_MLFLOW_LOCAL / "mlflow.db"

mlflow.set_tracking_uri(
    f"sqlite:///{RUTA_DB}"
)

# 5. Ruta de respaldo permanente en Google Drive
RUTA_MLFLOW_DRIVE = Path(
    "/content/drive_real/MyDrive/"
    "FinPredict-Actividad5/experimentos/mlflow"
)

RUTA_MLFLOW_DRIVE.mkdir(
    parents=True,
    exist_ok=True
)

# 6. Crear experimento
NOMBRE_EXPERIMENTO = "FinPredict_Actividad5"

experimento = mlflow.set_experiment(
    NOMBRE_EXPERIMENTO
)

print("\nFINPREDICT | Configuración MLflow")
print("-" * 60)
print("Drive montado:", os.path.ismount("/content/drive_real"))
print("Versión MLflow:", mlflow.__version__)
print("Experimento:", NOMBRE_EXPERIMENTO)
print("ID:", experimento.experiment_id)
print("Base de datos local:", RUTA_DB)
print("Destino de respaldo:", RUTA_MLFLOW_DRIVE)
print("-" * 60)
print("MLflow configurado. Sin nuevos entrenamientos.")


Modelos optimizados disponibles en memoria.
Mounted at /content/drive_real


2026/10/19 00:18:05 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/19 00:18:05 INFO mlflow.store.db.utils: Updating database tables
2026/10/19 00:18:08 INFO mlflow.tracking.fluent: Experiment with name 'FinPredict_Actividad5' does not exist. Creating a new experiment.



FINPREDICT | Configuración MLflow
------------------------------------------------------------
Drive montado: True
Versión MLflow: 3.17.1
Experimento: FinPredict_Actividad5
ID: 1
Base de datos local: /content/finpredict_mlflow/mlflow.db
Destino de respaldo: /content/drive_real/MyDrive/FinPredict-Actividad5/experimentos/mlflow
------------------------------------------------------------
MLflow configurado. Sin nuevos entrenamientos.



### 12. Registro de experimentos en MLflow

Los modelos optimizados se registran mediante MLflow Tracking para conservar evidencia reproducible de los experimentos.

Para cada algoritmo se documentan:

- Configuración seleccionada.
- Versión del dataset y número de predictores.
- Estrategia de validación temporal.
- Métricas promedio y por partición.
- Tiempo de optimización.
- Resultados completos de GridSearchCV.

Las métricas corresponden exclusivamente al conjunto de desarrollo y no representan el desempeño sobre la evaluación final reservada.


In [12]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 12
# Registro de experimentos optimizados en MLflow
# ============================================================

import mlflow
import pandas as pd
from pathlib import Path

# Carpeta local de artefactos
RUTA_ARTEFACTOS = Path(
    "/content/finpredict_mlflow/reportes"
)
RUTA_ARTEFACTOS.mkdir(parents=True, exist_ok=True)

experimentos_modelos = [
    {
        "nombre": "Regresion_Logistica_GridSearchCV",
        "algoritmo": "LogisticRegression",
        "busqueda": busqueda_logistica,
        "tiempo": tiempo_busqueda_logistica,
        "resultados": resultados_logistica_cv
    },
    {
        "nombre": "Random_Forest_GridSearchCV",
        "algoritmo": "RandomForestClassifier",
        "busqueda": busqueda_rf,
        "tiempo": tiempo_busqueda_rf,
        "resultados": resultados_rf_cv
    }
]

registros_mlflow = []

for experimento_modelo in experimentos_modelos:

    busqueda = experimento_modelo["busqueda"]

    # Identificar la configuración ganadora
    indice_mejor = busqueda.best_index_
    resultados = busqueda.cv_results_

    with mlflow.start_run(
        run_name=experimento_modelo["nombre"]
    ) as run:

        # Identificación y trazabilidad
        mlflow.set_tags({
            "proyecto": "FinPredict",
            "actividad": "5",
            "algoritmo": experimento_modelo["algoritmo"],
            "dataset": "finpredict_desarrollo_v1.parquet",
            "validacion": "temporal_expansiva",
            "estado": "modelo_optimizado"
        })

        # Parámetros de la configuración seleccionada
        mlflow.log_params({
            "numero_predictores": len(PREDICTORES),
            "folds": len(particiones),
            "metrica_optimizacion": "average_precision",
            **{
                clave.replace("modelo__", ""): str(valor)
                for clave, valor in busqueda.best_params_.items()
            }
        })

        # Métricas promedio del mejor candidato
        metricas = {
            "cv_pr_auc": float(
                resultados["mean_test_pr_auc"][indice_mejor]
            ),
            "cv_roc_auc": float(
                resultados["mean_test_roc_auc"][indice_mejor]
            ),
            "cv_recall": float(
                resultados["mean_test_recall"][indice_mejor]
            ),
            "cv_f1": float(
                resultados["mean_test_f1"][indice_mejor]
            ),
            "tiempo_busqueda_segundos": float(
                experimento_modelo["tiempo"]
            )
        }

        mlflow.log_metrics(metricas)

        # Métricas individuales de los tres folds
        for fold in range(len(particiones)):
            for metrica in ["pr_auc", "roc_auc", "recall", "f1"]:
                clave = f"split{fold}_test_{metrica}"
                mlflow.log_metric(
                    f"fold_{fold + 1}_{metrica}",
                    float(resultados[clave][indice_mejor])
                )

        # Exportar resultados completos del GridSearch
        nombre_csv = (
            experimento_modelo["nombre"] + "_resultados.csv"
        )

        ruta_csv = RUTA_ARTEFACTOS / nombre_csv

        experimento_modelo["resultados"].to_csv(
            ruta_csv,
            index=False
        )

        mlflow.log_artifact(
            str(ruta_csv),
            artifact_path="gridsearch"
        )

        registros_mlflow.append({
            "modelo": experimento_modelo["algoritmo"],
            "run_id": run.info.run_id,
            "pr_auc": metricas["cv_pr_auc"],
            "roc_auc": metricas["cv_roc_auc"],
            "recall": metricas["cv_recall"],
            "f1": metricas["cv_f1"]
        })

# Resumen de ejecuciones registradas
df_registros_mlflow = pd.DataFrame(registros_mlflow)

print("FINPREDICT | Experimentos MLflow")
print("-" * 65)

display(df_registros_mlflow.round(4))

print("\nExperimentos registrados:", len(df_registros_mlflow))
print("Nombre:", NOMBRE_EXPERIMENTO)

print("-" * 65)
print("Métricas, parámetros y resultados registrados.")
print("Modelos no reentrenados.")


FINPREDICT | Experimentos MLflow
-----------------------------------------------------------------


,modelo,run_id,pr_auc,roc_auc,recall,f1
0,LogisticRegression,a071b06d029642ef86912f15d578f5c2,0.8567,0.9844,0.6661,0.7452
1,RandomForestClassifier,9f1a167022214509af181b39c2d0e047,0.8631,0.9855,0.9284,0.6843



Experimentos registrados: 2
Nombre: FinPredict_Actividad5
-----------------------------------------------------------------
Métricas, parámetros y resultados registrados.
Modelos no reentrenados.


In [13]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 13
# Respaldo de experimentos MLflow en Google Drive
# ============================================================

from pathlib import Path
import shutil
import hashlib
import os
import json

# Validar que Google Drive esté montado
assert os.path.ismount("/content/drive_real"), (
    "Google Drive no está montado."
)

origen = Path("/content/finpredict_mlflow")

destino = Path(
    "/content/drive_real/MyDrive/"
    "FinPredict-Actividad5/experimentos/mlflow"
)

assert (origen / "mlflow.db").is_file(), (
    "No se encontró la base de experimentos."
)

# Cerrar conexiones antes de copiar SQLite
import mlflow

mlflow.end_run()

# Crear respaldo
destino.mkdir(parents=True, exist_ok=True)

# Respaldar base de datos mediante API de SQLite
import sqlite3

conexion_origen = sqlite3.connect(
    str(origen / "mlflow.db")
)

conexion_destino = sqlite3.connect(
    str(destino / "mlflow.db")
)

with conexion_destino:
    conexion_origen.backup(conexion_destino)

conexion_origen.close()
conexion_destino.close()

# Copiar artefactos y reportes
for nombre in ["reportes", "artifacts", "mlruns"]:
    carpeta_origen = origen / nombre

    if carpeta_origen.exists():
        shutil.copytree(
            carpeta_origen,
            destino / nombre,
            dirs_exist_ok=True
        )

# Verificar respaldo de la base de datos
assert (destino / "mlflow.db").is_file()

with sqlite3.connect(str(destino / "mlflow.db")) as con:
    comprobacion = con.execute(
        "PRAGMA integrity_check"
    ).fetchone()[0]

assert comprobacion == "ok"

print("FINPREDICT | Respaldo MLflow")
print("-" * 60)

print("Base de datos:", destino / "mlflow.db")
print("Integridad SQLite:", comprobacion)
print("Destino:", destino)

print("-" * 60)
print("Respaldo de experimentos completado.")


FINPREDICT | Respaldo MLflow
------------------------------------------------------------
Base de datos: /content/drive_real/MyDrive/FinPredict-Actividad5/experimentos/mlflow/mlflow.db
Integridad SQLite: ok
Destino: /content/drive_real/MyDrive/FinPredict-Actividad5/experimentos/mlflow
------------------------------------------------------------
Respaldo de experimentos completado.


In [14]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 14
# Verificación de experimentos y artefactos MLflow
# ============================================================

import mlflow
import pandas as pd
from pathlib import Path

# Recuperar las ejecuciones registradas
cliente = mlflow.tracking.MlflowClient()

runs = cliente.search_runs(
    experiment_ids=[experimento.experiment_id],
    max_results=100
)

registros = []

for run in runs:

    # Consultar artefactos del experimento
    artefactos = cliente.list_artifacts(
        run.info.run_id,
        path="gridsearch"
    )

    archivos_csv = [
        item.path
        for item in artefactos
        if item.path.endswith(".csv")
    ]

    registros.append({
        "modelo": run.data.tags.get("algoritmo"),
        "run_id": run.info.run_id,
        "pr_auc": run.data.metrics.get("cv_pr_auc"),
        "parametros": len(run.data.params),
        "csv_registrados": len(archivos_csv),
        "artifact_uri": run.info.artifact_uri
    })

df_auditoria_mlflow = pd.DataFrame(registros)

# Validar que las dos ejecuciones estén completas
assert len(df_auditoria_mlflow) == 2

assert df_auditoria_mlflow["pr_auc"].notna().all()
assert df_auditoria_mlflow["parametros"].gt(0).all()
assert df_auditoria_mlflow["csv_registrados"].eq(1).all()

print("FINPREDICT | Auditoría de experimentos MLflow")
print("-" * 65)

display(df_auditoria_mlflow)

print("-" * 65)
print("Dos ejecuciones con métricas, parámetros y CSV registrados.")


FINPREDICT | Auditoría de experimentos MLflow
-----------------------------------------------------------------


,modelo,run_id,pr_auc,parametros,csv_registrados,artifact_uri
0,RandomForestClassifier,9f1a167022214509af181b39c2d0e047,0.863065,5,1,/content/mlruns/1/9f1a167022214509af181b39c2d0...
1,LogisticRegression,a071b06d029642ef86912f15d578f5c2,0.856654,5,1,/content/mlruns/1/a071b06d029642ef86912f15d578...


-----------------------------------------------------------------
Dos ejecuciones con métricas, parámetros y CSV registrados.



### 15. Persistencia y respaldo de modelos optimizados

Se almacenan los pipelines optimizados de regresión logística y Random Forest, conservando el preprocesamiento y los hiperparámetros seleccionados.

Los modelos se vinculan a sus ejecuciones de MLflow y se respaldan junto con los resultados de GridSearchCV.

El objetivo es permitir su recuperación y utilización posterior sin repetir el proceso de optimización.

Los artefactos corresponden a modelos entrenados sobre el conjunto de desarrollo. La evaluación final permanece reservada.


In [15]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 15
# Persistencia de modelos y respaldo de artefactos
# ============================================================

from pathlib import Path
import joblib
import shutil
import hashlib
import os
import mlflow
import pandas as pd

# Verificar Drive
assert os.path.ismount("/content/drive_real")

# Directorios
RUTA_MODELOS_LOCAL = Path(
    "/content/finpredict_mlflow/modelos"
)

RUTA_MODELOS_DRIVE = Path(
    "/content/drive_real/MyDrive/"
    "FinPredict-Actividad5/experimentos/modelos_v1"
)

RUTA_ARTEFACTOS_DRIVE = Path(
    "/content/drive_real/MyDrive/"
    "FinPredict-Actividad5/experimentos/mlflow/mlruns"
)

for ruta in [
    RUTA_MODELOS_LOCAL,
    RUTA_MODELOS_DRIVE,
    RUTA_ARTEFACTOS_DRIVE
]:
    ruta.mkdir(parents=True, exist_ok=True)

# Modelos optimizados ya entrenados
modelos = [
    {
        "nombre": "regresion_logistica_optimizada.joblib",
        "modelo": busqueda_logistica.best_estimator_,
        "run_id": df_registros_mlflow.loc[
            df_registros_mlflow["modelo"] == "LogisticRegression",
            "run_id"
        ].iloc[0]
    },
    {
        "nombre": "random_forest_optimizado.joblib",
        "modelo": busqueda_rf.best_estimator_,
        "run_id": df_registros_mlflow.loc[
            df_registros_mlflow["modelo"] == "RandomForestClassifier",
            "run_id"
        ].iloc[0]
    }
]

resumen_modelos = []

for elemento in modelos:

    ruta_local = RUTA_MODELOS_LOCAL / elemento["nombre"]

    # Guardar pipeline completo, sin reentrenar
    joblib.dump(
        elemento["modelo"],
        ruta_local,
        compress=3
    )

    # Registrar modelo dentro de su ejecución MLflow
    with mlflow.start_run(run_id=elemento["run_id"]):
        mlflow.log_artifact(
            str(ruta_local),
            artifact_path="modelos"
        )

    # Respaldo adicional en Drive
    ruta_drive = RUTA_MODELOS_DRIVE / elemento["nombre"]
    shutil.copy2(ruta_local, ruta_drive)

    # Verificar que el archivo sea idéntico
    def sha256_archivo(ruta):
        sha = hashlib.sha256()
        with open(ruta, "rb") as archivo:
            for bloque in iter(
                lambda: archivo.read(1024 * 1024),
                b""
            ):
                sha.update(bloque)
        return sha.hexdigest()

    assert sha256_archivo(ruta_local) == sha256_archivo(
        ruta_drive
    )

    resumen_modelos.append({
        "modelo": elemento["nombre"],
        "tamano_mb": round(
            ruta_drive.stat().st_size / 1024**2,
            2
        ),
        "sha256_verificado": True
    })

# Respaldar la carpeta real de artefactos MLflow
ORIGEN_MLRUNS = Path("/content/mlruns")

assert ORIGEN_MLRUNS.exists(), (
    "No se encontró el directorio de artefactos MLflow."
)

shutil.copytree(
    ORIGEN_MLRUNS,
    RUTA_ARTEFACTOS_DRIVE,
    dirs_exist_ok=True
)

print("FINPREDICT | Persistencia de modelos")
print("-" * 65)

display(pd.DataFrame(resumen_modelos))

print("\nModelos:", RUTA_MODELOS_DRIVE)
print("Artefactos MLflow:", RUTA_ARTEFACTOS_DRIVE)

print("-" * 65)
print("Pipelines y artefactos respaldados.")
print("No se realizaron nuevos entrenamientos.")


FINPREDICT | Persistencia de modelos
-----------------------------------------------------------------


,modelo,tamano_mb,sha256_verificado
0,regresion_logistica_optimizada.joblib,0.00,True
1,random_forest_optimizado.joblib,7.24,True



Modelos: /content/drive_real/MyDrive/FinPredict-Actividad5/experimentos/modelos_v1
Artefactos MLflow: /content/drive_real/MyDrive/FinPredict-Actividad5/experimentos/mlflow/mlruns
-----------------------------------------------------------------
Pipelines y artefactos respaldados.
No se realizaron nuevos entrenamientos.



### 16. Comparación técnica de modelos optimizados

Se comparan los mejores modelos de regresión logística y Random Forest mediante las métricas promedio de tres particiones temporales expansivas.

La selección preliminar utiliza PR-AUC como criterio principal, dada la baja proporción de incumplimientos en el conjunto de desarrollo.

También se consideran recall, F1-score, ROC-AUC y costo computacional.

La comparación permite identificar un modelo candidato para la evaluación final, pero no constituye evidencia suficiente para declarar su superioridad en un entorno productivo.


In [16]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 16
# Comparación ejecutiva de modelos optimizados
# ============================================================

import pandas as pd

comparacion_final_cv = pd.DataFrame({
    "modelo": [
        "Regresión logística",
        "Random Forest"
    ],
    "pr_auc": [
        busqueda_logistica.best_score_,
        busqueda_rf.best_score_
    ],
    "roc_auc": [
        resultados_logistica_cv.loc[
            busqueda_logistica.best_index_,
            "mean_test_roc_auc"
        ],
        resultados_rf_cv.loc[
            busqueda_rf.best_index_,
            "mean_test_roc_auc"
        ]
    ],
    "recall": [
        resultados_logistica_cv.loc[
            busqueda_logistica.best_index_,
            "mean_test_recall"
        ],
        resultados_rf_cv.loc[
            busqueda_rf.best_index_,
            "mean_test_recall"
        ]
    ],
    "f1": [
        resultados_logistica_cv.loc[
            busqueda_logistica.best_index_,
            "mean_test_f1"
        ],
        resultados_rf_cv.loc[
            busqueda_rf.best_index_,
            "mean_test_f1"
        ]
    ],
    "tiempo_optimizacion_seg": [
        tiempo_busqueda_logistica,
        tiempo_busqueda_rf
    ]
})

# Candidato por la métrica de optimización
indice_ganador = comparacion_final_cv["pr_auc"].idxmax()

modelo_candidato = comparacion_final_cv.loc[
    indice_ganador, "modelo"
]

print("FINPREDICT | Comparación de modelos")
print("-" * 65)

display(comparacion_final_cv.round(4))

print("\nModelo candidato por PR-AUC:", modelo_candidato)

print("-" * 65)
print("Comparación de validación temporal completada.")
print("Evaluación final aún no utilizada.")


FINPREDICT | Comparación de modelos
-----------------------------------------------------------------


,modelo,pr_auc,roc_auc,recall,f1,tiempo_optimizacion_seg
0,Regresión logística,0.8567,0.9844,0.6661,0.7452,18.8258
1,Random Forest,0.8631,0.9855,0.9284,0.6843,1069.7665



Modelo candidato por PR-AUC: Random Forest
-----------------------------------------------------------------
Comparación de validación temporal completada.
Evaluación final aún no utilizada.


In [17]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 17
# Respaldo definitivo de MLflow y artefactos
# ============================================================

from pathlib import Path
import sqlite3
import shutil
import hashlib
import os
import pandas as pd
import mlflow

assert os.path.ismount("/content/drive_real"), (
    "Google Drive no está montado."
)

# Ubicaciones reales identificadas en la auditoría
RUTA_DB_LOCAL = Path(
    "/content/finpredict_mlflow/mlflow.db"
)

RUTA_DB_DRIVE = Path(
    "/content/drive_real/MyDrive/"
    "FinPredict-Actividad5/experimentos/mlflow/mlflow.db"
)

RUTA_ARTEFACTOS_LOCAL = Path("/content/mlruns")

RUTA_ARTEFACTOS_DRIVE = Path(
    "/content/drive_real/MyDrive/"
    "FinPredict-Actividad5/experimentos/mlflow/mlruns"
)

assert RUTA_DB_LOCAL.is_file()
assert RUTA_ARTEFACTOS_LOCAL.is_dir()

# Finalizar cualquier ejecución activa
mlflow.end_run()

# Crear respaldo consistente de SQLite
RUTA_DB_DRIVE.parent.mkdir(parents=True, exist_ok=True)

with sqlite3.connect(str(RUTA_DB_LOCAL)) as origen:
    with sqlite3.connect(str(RUTA_DB_DRIVE)) as destino:
        origen.backup(destino)

# Respaldar artefactos registrados
shutil.copytree(
    RUTA_ARTEFACTOS_LOCAL,
    RUTA_ARTEFACTOS_DRIVE,
    dirs_exist_ok=True
)

# Validar integridad de SQLite
with sqlite3.connect(str(RUTA_DB_DRIVE)) as conexion:
    integridad = conexion.execute(
        "PRAGMA integrity_check"
    ).fetchone()[0]

assert integridad == "ok"

# Verificar los artefactos de ambas ejecuciones
def sha256_archivo(ruta):
    sha = hashlib.sha256()
    with open(ruta, "rb") as archivo:
        for bloque in iter(
            lambda: archivo.read(1024 * 1024),
            b""
        ):
            sha.update(bloque)
    return sha.hexdigest()

cliente = mlflow.tracking.MlflowClient()
auditoria = []

for registro in df_registros_mlflow.itertuples(index=False):

    ejecucion = cliente.get_run(registro.run_id)

    ruta_local = Path(ejecucion.info.artifact_uri)

    for subcarpeta in ["gridsearch", "modelos"]:
        origen = ruta_local / subcarpeta

        assert origen.is_dir(), (
            f"Falta el artefacto {subcarpeta}: {registro.modelo}"
        )

        for archivo in origen.rglob("*"):
            if not archivo.is_file():
                continue

            ruta_relativa = archivo.relative_to(
                RUTA_ARTEFACTOS_LOCAL
            )

            archivo_drive = (
                RUTA_ARTEFACTOS_DRIVE / ruta_relativa
            )

            assert archivo_drive.is_file()
            assert sha256_archivo(archivo) == sha256_archivo(
                archivo_drive
            )

    auditoria.append({
        "modelo": registro.modelo,
        "run_id": registro.run_id,
        "integridad_artefactos": "VERIFICADA"
    })

print("FINPREDICT | Respaldo definitivo MLflow")
print("-" * 65)

display(pd.DataFrame(auditoria))

print("\nIntegridad SQLite:", integridad)
print("Respaldo:", RUTA_DB_DRIVE.parent)

print("-" * 65)
print("Base y artefactos respaldados correctamente.")


FINPREDICT | Respaldo definitivo MLflow
-----------------------------------------------------------------


,modelo,run_id,integridad_artefactos
0,LogisticRegression,a071b06d029642ef86912f15d578f5c2,VERIFICADA
1,RandomForestClassifier,9f1a167022214509af181b39c2d0e047,VERIFICADA



Integridad SQLite: ok
Respaldo: /content/drive_real/MyDrive/FinPredict-Actividad5/experimentos/mlflow
-----------------------------------------------------------------
Base y artefactos respaldados correctamente.



### 18. Evaluación final fuera de muestra

Se evalúa el modelo Random Forest seleccionado mediante validación temporal sobre el conjunto reservado de septiembre de 2025.

El modelo fue seleccionado utilizando únicamente las observaciones de desarrollo histórico, sin intervención del conjunto final.

La evaluación permitirá estimar su capacidad de generalización temporal mediante Precision, Recall, F1-score, ROC-AUC, PR-AUC y matriz de confusión.

Los resultados corresponden a datos sintéticos y no constituyen evidencia de desempeño en una cartera bancaria real.


In [18]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 18
# Evaluación final fuera de muestra | Random Forest
# ============================================================

import requests
import hashlib
import numpy as np
import pandas as pd

from io import BytesIO

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# 1. Cargar conjunto reservado
nombre_evaluacion = "finpredict_evaluacion_v1.parquet"

respuesta = requests.get(
    BASE_URL + nombre_evaluacion,
    timeout=180
)
respuesta.raise_for_status()

contenido = respuesta.content

# 2. Verificar SHA-256
sha256 = hashlib.sha256(contenido).hexdigest()

assert sha256 == manifiesto["archivos"][
    nombre_evaluacion
]["sha256"], "La integridad del archivo no coincide."

df_evaluacion_final = pd.read_parquet(
    BytesIO(contenido)
)

assert len(df_evaluacion_final) == 40_141

assert df_evaluacion_final[
    "uso"
].eq("EVALUACION_FINAL").all()

# 3. Construir matriz de evaluación
X_final = df_evaluacion_final[PREDICTORES].copy()

y_final = df_evaluacion_final[
    VARIABLE_OBJETIVO
].copy()

# 4. Retirar imputación estadística global de v1
mascara_mediana_final = (
    df_evaluacion_final["metodo_recuperacion_ingreso"]
    == "MEDIANA_TIPO_EMPLEO"
)

X_final.loc[
    mascara_mediana_final,
    "ingreso_mensual"
] = np.nan

X_final.loc[
    mascara_mediana_final,
    "relacion_pago_ingreso"
] = np.nan

# 5. Utilizar el modelo previamente seleccionado
modelo_final = busqueda_rf.best_estimator_

probabilidades_final = modelo_final.predict_proba(
    X_final
)[:, 1]

predicciones_final = (
    probabilidades_final >= 0.50
).astype(int)

# 6. Métricas finales
metricas_evaluacion_final = {
    "Precision": precision_score(
        y_final, predicciones_final, zero_division=0
    ),
    "Recall": recall_score(
        y_final, predicciones_final, zero_division=0
    ),
    "F1-score": f1_score(
        y_final, predicciones_final, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_final, probabilidades_final
    ),
    "PR-AUC": average_precision_score(
        y_final, probabilidades_final
    )
}

matriz_final = confusion_matrix(
    y_final, predicciones_final, labels=[0, 1]
)

print("FINPREDICT | Evaluación final Random Forest")
print("-" * 65)

print(f"Observaciones: {len(X_final):,}")
print(f"Incumplimientos reales: {int(y_final.sum()):,}")
print(f"Ingresos tratados por pipeline: {int(mascara_mediana_final.sum())}")
print("Umbral: 0.50")

print("\nMétricas finales:")
display(pd.DataFrame(
    metricas_evaluacion_final.items(),
    columns=["metrica", "valor"]
).round(4))

print("\nMatriz de confusión:")
display(pd.DataFrame(
    matriz_final,
    index=[
        "Real: sin incumplimiento",
        "Real: incumplimiento"
    ],
    columns=[
        "Predicho: sin incumplimiento",
        "Predicho: incumplimiento"
    ]
))

print("-" * 65)
print("Evaluación final completada sin reentrenamiento.")


FINPREDICT | Evaluación final Random Forest
-----------------------------------------------------------------
Observaciones: 40,141
Incumplimientos reales: 2,282
Ingresos tratados por pipeline: 0
Umbral: 0.50

Métricas finales:


,metrica,valor
0,Precision,0.5377
1,Recall,0.9321
2,F1-score,0.6819
3,ROC-AUC,0.9870
4,PR-AUC,0.8641



Matriz de confusión:


,Predicho: sin incumplimiento,Predicho: incumplimiento
Real: sin incumplimiento,36030,1829
Real: incumplimiento,155,2127


-----------------------------------------------------------------
Evaluación final completada sin reentrenamiento.


In [19]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 19
# Registro de evaluación final en MLflow
# ============================================================

import mlflow
import pandas as pd
from pathlib import Path

# Identificar ejecución del Random Forest optimizado
RUN_RF = df_registros_mlflow.loc[
    df_registros_mlflow["modelo"] == "RandomForestClassifier",
    "run_id"
].iloc[0]

# Matriz de confusión como artefacto
ruta_reporte = Path(
    "/content/finpredict_mlflow/reportes/"
    "matriz_confusion_evaluacion_final.csv"
)

ruta_reporte.parent.mkdir(parents=True, exist_ok=True)

pd.DataFrame(
    matriz_final,
    index=["Real_0", "Real_1"],
    columns=["Predicho_0", "Predicho_1"]
).to_csv(ruta_reporte)

# Registrar métricas finales en la ejecución existente
with mlflow.start_run(run_id=RUN_RF):

    mlflow.log_metrics({
        "final_precision": metricas_evaluacion_final["Precision"],
        "final_recall": metricas_evaluacion_final["Recall"],
        "final_f1": metricas_evaluacion_final["F1-score"],
        "final_roc_auc": metricas_evaluacion_final["ROC-AUC"],
        "final_pr_auc": metricas_evaluacion_final["PR-AUC"],
        "final_verdaderos_positivos": int(matriz_final[1, 1]),
        "final_falsos_negativos": int(matriz_final[1, 0]),
        "final_falsos_positivos": int(matriz_final[0, 1]),
        "final_verdaderos_negativos": int(matriz_final[0, 0])
    })

    mlflow.log_artifact(
        str(ruta_reporte),
        artifact_path="evaluacion_final"
    )

    mlflow.set_tag(
        "evaluacion_final",
        "completada_periodo_2025_Q4"
    )

print("FINPREDICT | Registro de evaluación final")
print("-" * 60)

print("Modelo: Random Forest optimizado")
print("Run ID:", RUN_RF)
print("PR-AUC final:", round(metricas_evaluacion_final["PR-AUC"], 4))
print("Observaciones evaluadas:", len(y_final))
print("Matriz de confusión registrada.")

print("-" * 60)
print("Evaluación final documentada en MLflow.")


FINPREDICT | Registro de evaluación final
------------------------------------------------------------
Modelo: Random Forest optimizado
Run ID: 9f1a167022214509af181b39c2d0e047
PR-AUC final: 0.8641
Observaciones evaluadas: 40141
Matriz de confusión registrada.
------------------------------------------------------------
Evaluación final documentada en MLflow.


In [20]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 20
# Respaldo final de MLflow
# ============================================================

from pathlib import Path
import sqlite3
import shutil
import hashlib
import os
import mlflow

assert os.path.ismount("/content/drive_real"), (
    "Google Drive no está montado."
)

mlflow.end_run()

base_local = Path(
    "/content/finpredict_mlflow/mlflow.db"
)

base_drive = Path(
    "/content/drive_real/MyDrive/FinPredict-Actividad5/"
    "experimentos/mlflow/mlflow.db"
)

artefactos_local = Path("/content/mlruns")

artefactos_drive = Path(
    "/content/drive_real/MyDrive/FinPredict-Actividad5/"
    "experimentos/mlflow/mlruns"
)

# 1. Respaldo consistente de SQLite
base_drive.parent.mkdir(parents=True, exist_ok=True)

with sqlite3.connect(str(base_local)) as origen:
    with sqlite3.connect(str(base_drive)) as destino:
        origen.backup(destino)

# 2. Copia de artefactos
shutil.copytree(
    artefactos_local,
    artefactos_drive,
    dirs_exist_ok=True
)

# 3. Verificar integridad de SQLite
with sqlite3.connect(str(base_drive)) as conexion:
    integridad = conexion.execute(
        "PRAGMA integrity_check"
    ).fetchone()[0]

assert integridad == "ok"

# 4. Verificar el artefacto de evaluación final
cliente = mlflow.tracking.MlflowClient()

artefactos_finales = cliente.list_artifacts(
    RUN_RF,
    path="evaluacion_final"
)

assert any(
    archivo.path.endswith(
        "matriz_confusion_evaluacion_final.csv"
    )
    for archivo in artefactos_finales
)

# 5. Verificar que el archivo también está en Drive
ruta_run = Path(
    cliente.get_run(RUN_RF).info.artifact_uri
)

archivo_local = (
    ruta_run
    / "evaluacion_final"
    / "matriz_confusion_evaluacion_final.csv"
)

archivo_drive = (
    artefactos_drive
    / archivo_local.relative_to(artefactos_local)
)

assert archivo_local.is_file()
assert archivo_drive.is_file()

assert archivo_local.read_bytes() == archivo_drive.read_bytes()

print("FINPREDICT | Respaldo final MLflow")
print("-" * 60)
print("Integridad SQLite:", integridad)
print("Matriz de confusión: RESPALDADA")
print("Run ID:", RUN_RF)
print("Ruta:", base_drive.parent)
print("-" * 60)
print("Experimentos y evaluación final respaldados.")


FINPREDICT | Respaldo final MLflow
------------------------------------------------------------
Integridad SQLite: ok
Matriz de confusión: RESPALDADA
Run ID: 9f1a167022214509af181b39c2d0e047
Ruta: /content/drive_real/MyDrive/FinPredict-Actividad5/experimentos/mlflow
------------------------------------------------------------
Experimentos y evaluación final respaldados.


In [21]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 21
# Visualización de MLflow Tracking en Colab
# ============================================================

import subprocess
import time
from IPython.display import display, HTML
from google.colab import output

# Iniciar servidor MLflow usando la base de la sesión
proceso_mlflow_ui = subprocess.Popen(
    [
        "mlflow", "ui",
        "--backend-store-uri",
        "sqlite:////content/finpredict_mlflow/mlflow.db",
        "--host", "0.0.0.0",
        "--port", "5000"
    ],
    stdout=open("/content/mlflow_ui.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("FINPREDICT | Interfaz MLflow")
print("-" * 60)
print("Servidor iniciado:", proceso_mlflow_ui.poll() is None)
print("Experimento: FinPredict_Actividad5")

# Abrir interfaz mediante proxy de Google Colab
output.serve_kernel_port_as_window(5000)


FINPREDICT | Interfaz MLflow
------------------------------------------------------------
Servidor iniciado: True
Experimento: FinPredict_Actividad5
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [22]:
from google.colab import output

output.serve_kernel_port_as_iframe(5000)

<IPython.core.display.Javascript object>

In [23]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 22
# Acceso seguro a MLflow desde Google Colab
# ============================================================

import os
import subprocess
import time
from google.colab import output

# 1. Detener únicamente el servidor anterior
if "proceso_mlflow_ui" in globals():
    if proceso_mlflow_ui.poll() is None:
        proceso_mlflow_ui.terminate()
        try:
            proceso_mlflow_ui.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proceso_mlflow_ui.kill()
            proceso_mlflow_ui.wait()

# 2. Autorizar el dominio del proxy de Google Colab
entorno = os.environ.copy()
entorno["MLFLOW_SERVER_ALLOWED_HOSTS"] = (
    "localhost,127.0.0.1,*.colab.dev"
)

# 3. Reiniciar la interfaz usando la misma base de datos
proceso_mlflow_ui = subprocess.Popen(
    [
        "mlflow", "ui",
        "--backend-store-uri",
        "sqlite:////content/finpredict_mlflow/mlflow.db",
        "--host", "0.0.0.0",
        "--port", "5000"
    ],
    env=entorno,
    stdout=open("/content/mlflow_ui.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

assert proceso_mlflow_ui.poll() is None, (
    "El servidor no inició. Revisar /content/mlflow_ui.log"
)

print("FINPREDICT | Interfaz MLflow")
print("-" * 60)
print("Servidor activo: True")
print("Base de experimentos: conservada")
print("Host de Colab: autorizado")
print("-" * 60)

# 4. Mostrar interfaz dentro del notebook
output.serve_kernel_port_as_iframe(5000)


FINPREDICT | Interfaz MLflow
------------------------------------------------------------
Servidor activo: True
Base de experimentos: conservada
Host de Colab: autorizado
------------------------------------------------------------


<IPython.core.display.Javascript object>

In [24]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 23
# Reinicio de interfaz MLflow para proxy de Colab
# ============================================================

import subprocess
import time
from google.colab import output

# 1. Detener el servidor anterior
if "proceso_mlflow_ui" in globals():
    if proceso_mlflow_ui.poll() is None:
        proceso_mlflow_ui.terminate()
        try:
            proceso_mlflow_ui.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proceso_mlflow_ui.kill()
            proceso_mlflow_ui.wait()

# 2. Iniciar interfaz sin comprobación de hosts
# Solo para el entorno de prueba de Colab
proceso_mlflow_ui = subprocess.Popen(
    [
        "mlflow", "ui",
        "--backend-store-uri",
        "sqlite:////content/finpredict_mlflow/mlflow.db",
        "--host", "0.0.0.0",
        "--port", "5000",
        "--disable-security-middleware"
    ],
    stdout=open("/content/mlflow_ui.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

# 3. Comprobar inicio
if proceso_mlflow_ui.poll() is not None:
    print("El servidor no inició. Últimas líneas:")
    print(open("/content/mlflow_ui.log").read()[-2500:])
else:
    print("FINPREDICT | Interfaz MLflow")
    print("-" * 55)
    print("Servidor activo: True")
    print("Experimentos conservados")
    print("-" * 55)

    output.serve_kernel_port_as_iframe(5000)


FINPREDICT | Interfaz MLflow
-------------------------------------------------------
Servidor activo: True
Experimentos conservados
-------------------------------------------------------


<IPython.core.display.Javascript object>

In [25]:

# ============================================================
# FINPREDICT | ACTIVIDAD 5
# NOTEBOOK 03 - CELDA 24
# Abrir interfaz de MLflow en ventana independiente
# ============================================================

from google.colab import output

# Mostrar enlace de acceso al servidor existente
url_mlflow = output.eval_js(
    "google.colab.kernel.proxyPort(5000)"
)

print("FINPREDICT | Acceso a MLflow")
print("-" * 55)
print("Abre este enlace en una pestaña nueva:")
print(url_mlflow)
print("-" * 55)


FINPREDICT | Acceso a MLflow
-------------------------------------------------------
Abre este enlace en una pestaña nueva:
https://5000-m-s-kkb-usw1b1-1g3xouxvz30mk-b.us-west1-1.prod.colab.dev
-------------------------------------------------------
